# Lighthouses of Spain
## Day 01 · Points · #30DayMapChallenge 2026

Lighthouse locations from OpenStreetMap, displayed over a shaded
Copernicus DEM background adapted from my 2025 QGIS project.

The animation uses illustrative light pulses.
These do not represent the actual flashing characteristics of each lighthouse.

**Map by Maria Victoria León Parra**

### Required local files

Place these files in the project's `data/` directory:

- `dem_espana_3035_reproj.tif`
- `hillshade_espana.tif`

The raster coordinates are interpreted as EPSG:3035, matching the
original QGIS project. The source files are not modified.

In [ ]:
# Run once if required, then restart the kernel.
%pip install numpy pandas geopandas osmnx matplotlib rasterio shapely pillow pyproj

In [ ]:
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import rasterio

from rasterio.enums import Resampling
from matplotlib import pyplot as plt
from matplotlib.colors import (
    LinearSegmentedColormap,
    Normalize,
    to_rgb,
)
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

# Locate the project directory.
ROOT = Path.cwd()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent

DATA = ROOT / "data"
CACHE = ROOT / "cache"
OUTPUTS = ROOT / "outputs"

CACHE.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

DEM_PATH = DATA / "dem_espana_3035_reproj.tif"
HILLSHADE_PATH = DATA / "hillshade_espana.tif"

for path in (DEM_PATH, HILLSHADE_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Required raster not found: {path}")

# Map appearance.
MAP_CRS = "EPSG:3035"
SEA = "#081829"
PAPER = "#fff9e8"
TEXT = "#0c223f"
LIGHT = "#fff0a8"
HALO = "#fff4bf"

FIGSIZE = (14, 10)
DPI = 100

# Read a reduced version for display.
RASTER_WIDTH = 2400

# Animation settings: two complete cycles for a seamless loop.
FPS = 15
DURATION = 6
PULSE_PERIOD = 3
TOTAL_FRAMES = FPS * DURATION

# Cache OpenStreetMap requests.
ox.settings.use_cache = True
ox.settings.cache_folder = str(CACHE / "osmnx")
ox.settings.log_console = False
ox.settings.max_query_area_size = 25e12

print(f"Project directory: {ROOT}")

In [ ]:
def read_display_raster(path, target_width=RASTER_WIDTH):
    """Read a reduced raster without modifying the source file."""
    with rasterio.open(path) as src:
        width = min(target_width, src.width)
        height = max(1, round(src.height * width / src.width))

        array = src.read(
            1,
            out_shape=(height, width),
            masked=True,
            resampling=Resampling.bilinear,
        )

        extent = (
            src.bounds.left,
            src.bounds.right,
            src.bounds.bottom,
            src.bounds.top,
        )

        print(f"{path.name}: {src.width} × {src.height}")
        print(f"Display array: {width} × {height}")
        print(f"Stored CRS: {src.crs}\n")

    return array, extent


dem, dem_extent = read_display_raster(DEM_PATH)
hillshade, hillshade_extent = read_display_raster(HILLSHADE_PATH)

if dem.shape != hillshade.shape:
    raise ValueError("The display rasters have different shapes.")

if not np.allclose(dem_extent, hillshade_extent):
    raise ValueError("The raster extents do not match.")

In [ ]:
BOUNDARY_CACHE = CACHE / "spain_boundary.geojson"
LIGHTS_CACHE = CACHE / "spain_lighthouses.geojson"

# Load or download Spain's boundary.
if BOUNDARY_CACHE.exists():
    spain = gpd.read_file(BOUNDARY_CACHE)
else:
    spain = ox.geocode_to_gdf("Spain")
    spain.to_file(
        BOUNDARY_CACHE,
        driver="GeoJSON",
        index=False,
    )

# Follow the tutorial's two lighthouse queries.
if LIGHTS_CACHE.exists():
    lights = gpd.read_file(LIGHTS_CACHE)

else:
    tags_A = {"man_made": "lighthouse"}
    gdf_A = ox.features.features_from_place("Spain", tags_A)

    tags_B = {
        "seamark:type": [
            "light_major",
            "light_minor",
            "lighthouse",
        ]
    }
    gdf_B = ox.features.features_from_place("Spain", tags_B)

    print(f"Lighthouse structures: {len(gdf_A)}")
    print(f"Seamark features: {len(gdf_B)}")

    def normalize_points(gdf):
        columns = [
            column
            for column in ["name", "seamark:type", "man_made"]
            if column in gdf.columns
        ]

        result = gdf[columns + ["geometry"]].copy()
        result = result.loc[
            result.geometry.notna()
            & ~result.geometry.is_empty
        ].copy()

        result.geometry = result.geometry.apply(
            lambda geometry: (
                geometry
                if geometry.geom_type == "Point"
                else geometry.representative_point()
            )
        )
        return result

    points_A = normalize_points(gdf_A)
    points_B = normalize_points(gdf_B)

    lights = gpd.GeoDataFrame(
        pd.concat([points_A, points_B], ignore_index=True),
        geometry="geometry",
        crs=points_A.crs,
    )

    lights["_geometry_key"] = lights.geometry.apply(
        lambda geometry: geometry.wkb_hex
    )

    lights = (
        lights.drop_duplicates("_geometry_key")
        .drop(columns="_geometry_key")
        .reset_index(drop=True)
    )

    if lights.empty:
        raise RuntimeError("No lighthouse points were returned.")

    lights.to_file(
        LIGHTS_CACHE,
        driver="GeoJSON",
        index=False,
    )

lights_3035 = lights.to_crs(MAP_CRS)

print(f"Total locations: {len(lights)}")
lights.head()

In [ ]:
# Elevation colours recovered from the original QGIS project.
elevations = np.array([-5, 0, 200, 800, 1500, 2500, 3277])
elevation_colours = [
    "#2d2c55",
    "#0c223f",
    "#4d3a30",
    "#73513a",
    "#a26f45",
    "#d3955c",
    "#f7c67a",
]

elevation_positions = (elevations + 5) / 3282

terrain_cmap = LinearSegmentedColormap.from_list(
    "terrain_2025",
    list(zip(elevation_positions, elevation_colours)),
)

shade_values = np.array([
    1, 34.02, 67.04, 100.06, 133.08,
    166.1, 199.12, 229.6, 255,
])
shade_colours = [
    "#0c223f",
    "#0e59a2",
    "#2b7bba",
    "#4f9bcb",
    "#7ab6d9",
    "#a9cfe5",
    "#cde0f1",
    "#e4eff9",
    "#f7fbff",
]

shade_cmap = LinearSegmentedColormap.from_list(
    "shade_2025",
    list(zip((shade_values - 1) / 254, shade_colours)),
)

# Display mask: elevation <= 0 is treated as sea/background.
# This also hides any land below sea level.
elevation_data = dem.filled(0).astype(float)

land_mask = (
    ~np.ma.getmaskarray(dem)
    & np.isfinite(elevation_data)
    & (elevation_data > 0)
)

sea_rgb = np.array(to_rgb(SEA))

terrain_rgb = terrain_cmap(
    Normalize(-5, 3277)(elevation_data)
)[..., :3]

base_rgb = 0.75 * terrain_rgb + 0.25 * sea_rgb

# Approximate the original hillshade brightness and contrast.
shade = np.clip(
    (hillshade.filled(1).astype(float) - 1) / 254,
    0,
    1,
)

contrast = 17
brightness = 10
contrast_factor = (
    259 * (contrast + 255)
    / (255 * (259 - contrast))
)

shade = np.clip(
    contrast_factor * (shade - 0.5)
    + 0.5
    + brightness / 255,
    0,
    1,
)

shade_rgb = shade_cmap(shade)[..., :3]

# Approximate the QGIS blend, rather than an exact rendering match.
background_rgb = base_rgb * (0.60 + 0.40 * shade_rgb)
background_rgb[~land_mask] = sea_rgb
background_rgb = np.clip(background_rgb, 0, 1)

In [ ]:
parts = spain.to_crs(4326).explode(
    index_parts=False
).reset_index(drop=True)

bounds = parts.geometry.bounds

mainland_islands = parts.loc[
    bounds["miny"] > 35.5
].to_crs(MAP_CRS)

canary_islands = parts.loc[
    bounds["maxy"] < 30
].to_crs(MAP_CRS)

if mainland_islands.empty or canary_islands.empty:
    raise RuntimeError("Could not separate the map regions.")

light_latitudes = lights.to_crs(4326).geometry.y.to_numpy()

main_lights = lights_3035.loc[
    light_latitudes > 35.5
].copy()

canary_lights = lights_3035.loc[
    light_latitudes < 30
].copy()

print(f"Main map: {len(main_lights)} locations")
print(f"Canary Islands: {len(canary_lights)} locations")

In [ ]:
def draw_background(axis):
    axis.set_facecolor(SEA)
    axis.imshow(
        background_rgb,
        extent=dem_extent,
        origin="upper",
        interpolation="bilinear",
        zorder=0,
    )

    axis.set_aspect("equal", adjustable="box")
    axis.set_xticks([])
    axis.set_yticks([])

    for spine in axis.spines.values():
        spine.set_color("#56616c")
        spine.set_linewidth(1.5)


def fit_extent(axis, bounds, aspect_ratio, margin=0.05):
    xmin, ymin, xmax, ymax = bounds

    centre_x = (xmin + xmax) / 2
    centre_y = (ymin + ymax) / 2

    width = (xmax - xmin) * (1 + 2 * margin)
    height = (ymax - ymin) * (1 + 2 * margin)

    width = max(width, height * aspect_ratio)
    height = width / aspect_ratio

    axis.set_xlim(centre_x - width / 2, centre_x + width / 2)
    axis.set_ylim(centre_y - height / 2, centre_y + height / 2)


def draw_lights(axis, points):
    x = points.geometry.x.to_numpy()
    y = points.geometry.y.to_numpy()

    halos = []
    halo_opacities = [0.025, 0.045, 0.08, 0.15]

    for size, opacity in zip(
        [180, 100, 50, 22],
        halo_opacities,
    ):
        artist = axis.scatter(
            x, y,
            s=size,
            c=HALO,
            alpha=opacity,
            edgecolors="none",
            zorder=5,
        )
        halos.append(artist)

    cores = axis.scatter(
        x, y,
        s=9,
        c=LIGHT,
        alpha=0.90,
        edgecolors="none",
        zorder=6,
    )

    return {
        "halos": halos,
        "halo_opacities": halo_opacities,
        "cores": cores,
        "count": len(points),
    }


def create_map():
    figure = plt.figure(
        figsize=FIGSIZE,
        dpi=DPI,
        facecolor=PAPER,
    )

    left, bottom, width, height = 0.04, 0.05, 0.92, 0.85
    axis = figure.add_axes([left, bottom, width, height])

    draw_background(axis)

    aspect = (
        FIGSIZE[0] * width
        / (FIGSIZE[1] * height)
    )
    fit_extent(axis, mainland_islands.total_bounds, aspect)

    inset_width, inset_height = 0.36, 0.25
    inset = axis.inset_axes([
        1 - inset_width,
        1 - inset_height,
        inset_width,
        inset_height,
    ])

    draw_background(inset)
    fit_extent(
        inset,
        canary_islands.total_bounds,
        aspect * inset_width / inset_height,
        margin=0.08,
    )

    light_groups = [
        draw_lights(axis, main_lights),
        draw_lights(inset, canary_lights),
    ]

    figure.suptitle(
        "Day 1 – Points | Lighthouses of Spain",
        fontsize=23,
        fontweight="bold",
        color=TEXT,
        y=0.96,
    )

    figure.text(
        0.5, 0.022,
        "#30DayMapChallenge | 2026 | Day - 01 | "
        "Lighthouses of Spain | Map by Maria Victoria León Parra",
        ha="center",
        va="center",
        fontsize=10,
        color=TEXT,
    )

    return figure, light_groups

In [ ]:
preview_figure, _ = create_map()
preview_path = OUTPUTS / "day01_lighthouses_spain.png"

with plt.rc_context({"savefig.bbox": None}):
    preview_figure.savefig(
        preview_path,
        dpi=DPI,
        facecolor=PAPER,
        bbox_inches=None,
    )

plt.close(preview_figure)
display(Image(filename=str(preview_path)))

print(f"Saved: {preview_path}")

In [ ]:
animation_figure, light_groups = create_map()

# Reproducible phase offsets: lights do not pulse simultaneously.
rng = np.random.default_rng(2026)

for group in light_groups:
    group["phase"] = rng.uniform(
        0, 2 * np.pi, group["count"]
    )


def update(frame):
    time = frame / FPS
    artists = []

    for group in light_groups:
        pulse = 0.5 * (
            1 + np.sin(
                2 * np.pi * time / PULSE_PERIOD
                + group["phase"]
            )
        )

        for halo, maximum in zip(
            group["halos"],
            group["halo_opacities"],
        ):
            halo.set_alpha(maximum * (0.15 + 0.85 * pulse))
            artists.append(halo)

        group["cores"].set_alpha(0.15 + 0.75 * pulse)
        artists.append(group["cores"])

    return artists


animation = FuncAnimation(
    animation_figure,
    update,
    frames=TOTAL_FRAMES,
    interval=1000 / FPS,
    blit=False,
    repeat=True,
)

gif_path = OUTPUTS / "day01_lighthouses_spain.gif"

try:
    with plt.rc_context({"savefig.bbox": None}):
        animation.save(
            gif_path,
            writer=PillowWriter(fps=FPS),
            dpi=DPI,
        )
finally:
    plt.close(animation_figure)

display(Image(filename=str(gif_path)))
print(f"Saved: {gif_path}")